# Librerias 

In [57]:
import pandas as pd 
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Datos 

In [76]:
df_ml = pd.read_csv(r"D:\\Datases_CD\\7mo\\Aprendizaje_Automatico\\Proyecto\\datos_preprocesados_finales_2y.csv")
df_ml.head()

,Date,Close,Ticker,Return,ret_5d,ret_20d,vol_5d,vol_20d,ma_20,ma_60,dist_ma_20,dist_ma_60,rsi_10,stoch_k_10,spy_ret_20d,excess_ret_20d_vs_spy,categoria
0,2023-11-24,188.192719,AAPL,-0.007004,0.001370,0.139794,0.006422,0.009440,180.838142,176.714100,0.040669,0.064956,75.423679,77.744321,0.103624,0.036171,Technology
1,2023-11-27,188.014404,AAPL,-0.000948,0.000527,0.129711,0.006446,0.009582,181.917516,176.749881,0.033515,0.063731,65.013281,75.037581,0.106652,0.023059,Technology
2,2023-11-28,188.618698,AAPL,0.003214,-0.005484,0.119566,0.004599,0.009491,182.924707,176.769517,0.031128,0.067032,77.184361,73.815395,0.094660,0.024905,Technology
3,2023-11-29,187.598328,AAPL,-0.005410,-0.006662,0.110379,0.004825,0.009796,183.857136,176.768189,0.020348,0.061267,61.104573,39.534780,0.087064,0.023316,Technology
4,2023-11-30,188.172897,AAPL,0.003063,-0.007109,0.093293,0.004714,0.009277,184.659998,176.888398,0.019024,0.063794,61.149333,27.884331,0.079828,0.013466,Technology


In [59]:
RANDOM_STATE = 42

df_ml["Date"] = pd.to_datetime(df_ml["Date"])
df_ml = df_ml.sort_values(["Ticker", "Date"]).reset_index(drop=True)

TARGET_COL = "ret_5d"
cols_excluir = ["Date", "Ticker", TARGET_COL, "categoria"]
feature_cols = [c for c in df_ml.columns if c not in cols_excluir]

print("Features usadas:", feature_cols)

Features usadas: ['Close', 'Return', 'ret_20d', 'vol_5d', 'vol_20d', 'ma_20', 'ma_60', 'dist_ma_20', 'dist_ma_60', 'rsi_10', 'stoch_k_10', 'spy_ret_20d', 'excess_ret_20d_vs_spy']


In [60]:
def preparar_datos_ticker(df, ticker, feature_cols, target_col="ret_5d", test_size_frac=0.2):
    df_ticker = df[df["Ticker"] == ticker].sort_values("Date").reset_index(drop=True)

    if df_ticker.empty:
        raise ValueError(f"No hay datos para el ticker {ticker}")

    X = df_ticker[feature_cols].values
    y = df_ticker[target_col].values

    n = len(df_ticker)
    test_size = max(1, int(np.floor(n * test_size_frac)))
    train_size = n - test_size

    if train_size < 10:
        raise ValueError(f"Muy pocos datos de entrenamiento para {ticker}: {train_size} filas")

    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]

    return X_train, X_test, y_train, y_test, df_ticker

In [61]:
try:
    from sklearn.metrics import root_mean_squared_error

    def rmse(y_true, y_pred):
        return root_mean_squared_error(y_true, y_pred)

except ImportError:
    # Para versiones más viejas de sklearn
    from sklearn.metrics import mean_squared_error

    def rmse(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)


# Elastic Net 

In [62]:
def entrenar_elasticnet_time_series(X_train, y_train, X_test, y_test,
                                    n_splits=5,
                                    alpha=0.001,
                                    l1_ratio=0.7):
    tscv = TimeSeriesSplit(n_splits=n_splits)

    pipe_en = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=alpha,
                             l1_ratio=l1_ratio,
                             random_state=RANDOM_STATE,
                             max_iter=10000))
    ])

    # Validación cruzada temporal (solo train)
    cv_mae = -cross_val_score(pipe_en, X_train, y_train,
                              cv=tscv,
                              scoring="neg_mean_absolute_error")

    # Entrenamos en todo el train
    pipe_en.fit(X_train, y_train)

    # Métricas en test (último bloque temporal)
    y_pred = pipe_en.predict(X_test)
    test_mae = mean_absolute_error(y_test, y_pred)
    test_rmse = rmse(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    resultados = {
        "modelo": pipe_en,
        "cv_mae_mean": cv_mae.mean(),
        "cv_mae_std": cv_mae.std(),
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2
    }
    return resultados

In [63]:
def correr_elasticnet_en_todos_los_tickers(df,
                                           feature_cols,
                                           target_col="ret_5d",
                                           test_size_frac=0.2,
                                           n_splits=5,
                                           alpha=0.001,
                                           l1_ratio=0.7):
    resultados = {}
    
    tickers = df["Ticker"].unique()
    print(f"Voy a entrenar ElasticNet para {len(tickers)} tickers")
    
    for ticker in tickers:
        try:
            X_train, X_test, y_train, y_test, df_ticker = preparar_datos_ticker(
                df,
                ticker=ticker,
                feature_cols=feature_cols,
                target_col=target_col,
                test_size_frac=test_size_frac
            )
            
            res = entrenar_elasticnet_time_series(
                X_train, y_train, X_test, y_test,
                n_splits=n_splits,
                alpha=alpha,
                l1_ratio=l1_ratio
            )
            
            resultados[ticker] = res
            
            print(f"[ElasticNet] {ticker}: "
                  f"CV_MAE={res['cv_mae_mean']:.5f}, "
                  f"Test_MAE={res['test_mae']:.5f}")
        
        except ValueError as e:
            # Por si algún ticker tiene muy pocos datos
            print(f"Saltando {ticker}: {e}")
    
    return resultados

In [64]:
resultados_en = correr_elasticnet_en_todos_los_tickers(
    df_ml,
    feature_cols=feature_cols,
    target_col=TARGET_COL,
    test_size_frac=0.2,
    n_splits=5,
    alpha=0.001,
    l1_ratio=0.7
)

Voy a entrenar ElasticNet para 116 tickers
[ElasticNet] AAPL: CV_MAE=0.01786, Test_MAE=0.01574
[ElasticNet] ABBV: CV_MAE=0.01832, Test_MAE=0.01625
[ElasticNet] ABT: CV_MAE=0.01616, Test_MAE=0.01032
[ElasticNet] ACN: CV_MAE=0.01655, Test_MAE=0.01720
[ElasticNet] ADBE: CV_MAE=0.02396, Test_MAE=0.01543
[ElasticNet] ADI: CV_MAE=0.02182, Test_MAE=0.01453
[ElasticNet] AMAT: CV_MAE=0.02565, Test_MAE=0.02262
[ElasticNet] AMD: CV_MAE=0.03147, Test_MAE=0.03706
[ElasticNet] AMGN: CV_MAE=0.01548, Test_MAE=0.01594
[ElasticNet] AMZN: CV_MAE=0.02059, Test_MAE=0.01794
[ElasticNet] ANET: CV_MAE=0.03004, Test_MAE=0.02481
[ElasticNet] APH: CV_MAE=0.01956, Test_MAE=0.01569
[ElasticNet] AVGO: CV_MAE=0.04426, Test_MAE=0.02401
[ElasticNet] AXP: CV_MAE=0.01860, Test_MAE=0.01468
[ElasticNet] BA: CV_MAE=0.02392, Test_MAE=0.01645
[ElasticNet] BAC: CV_MAE=0.01743, Test_MAE=0.01205
[ElasticNet] BKNG: CV_MAE=0.01630, Test_MAE=0.01365
[ElasticNet] BLK: CV_MAE=0.01585, Test_MAE=0.01169
[ElasticNet] BRK-B: CV_MAE=0.01

# GLM

In [65]:
def entrenar_glm_ridge_time_series(X_train, y_train, X_test, y_test,
                                   n_splits=5,
                                   alpha=1.0):
    tscv = TimeSeriesSplit(n_splits=n_splits)

    pipe_ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha, random_state=RANDOM_STATE))
    ])

    from sklearn.model_selection import cross_val_score
    cv_mae = -cross_val_score(pipe_ridge, X_train, y_train,
                              cv=tscv,
                              scoring="neg_mean_absolute_error")

    pipe_ridge.fit(X_train, y_train)

    y_pred = pipe_ridge.predict(X_test)
    test_mae = mean_absolute_error(y_test, y_pred)
    test_rmse = rmse(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    resultados = {
        "modelo": pipe_ridge,
        "cv_mae_mean": cv_mae.mean(),
        "cv_mae_std": cv_mae.std(),
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2
    }
    return resultados


In [66]:
def correr_glm_ridge_en_todos_los_tickers(df,
                                          feature_cols,
                                          target_col="ret_5d",
                                          test_size_frac=0.2,
                                          n_splits=5,
                                          alpha=1.0):
    resultados = {}
    
    tickers = df["Ticker"].unique()
    print(f"Voy a entrenar GLM Ridge para {len(tickers)} tickers")
    
    for ticker in tickers:
        try:
            X_train, X_test, y_train, y_test, df_ticker = preparar_datos_ticker(
                df,
                ticker=ticker,
                feature_cols=feature_cols,
                target_col=target_col,
                test_size_frac=test_size_frac
            )
            
            res = entrenar_glm_ridge_time_series(
                X_train, y_train, X_test, y_test,
                n_splits=n_splits,
                alpha=alpha
            )
            
            resultados[ticker] = res
            
            print(f"[Ridge] {ticker}: "
                  f"CV_MAE={res['cv_mae_mean']:.5f}, "
                  f"Test_MAE={res['test_mae']:.5f}")
        
        except ValueError as e:
            print(f"Saltando {ticker}: {e}")
    
    return resultados

In [67]:
resultados_ridge = correr_glm_ridge_en_todos_los_tickers(
    df_ml,
    feature_cols=feature_cols,
    target_col=TARGET_COL,
    test_size_frac=0.2,
    n_splits=5,
    alpha=1.0
)

Voy a entrenar GLM Ridge para 116 tickers
[Ridge] AAPL: CV_MAE=0.01747, Test_MAE=0.01543
[Ridge] ABBV: CV_MAE=0.01893, Test_MAE=0.01611
[Ridge] ABT: CV_MAE=0.02054, Test_MAE=0.01023
[Ridge] ACN: CV_MAE=0.01628, Test_MAE=0.01734
[Ridge] ADBE: CV_MAE=0.02408, Test_MAE=0.01569
[Ridge] ADI: CV_MAE=0.02299, Test_MAE=0.01437
[Ridge] AMAT: CV_MAE=0.02603, Test_MAE=0.02278
[Ridge] AMD: CV_MAE=0.03358, Test_MAE=0.03615
[Ridge] AMGN: CV_MAE=0.01679, Test_MAE=0.01628
[Ridge] AMZN: CV_MAE=0.02175, Test_MAE=0.01846
[Ridge] ANET: CV_MAE=0.03127, Test_MAE=0.02599
[Ridge] APH: CV_MAE=0.01981, Test_MAE=0.01598
[Ridge] AVGO: CV_MAE=0.04564, Test_MAE=0.02414
[Ridge] AXP: CV_MAE=0.02002, Test_MAE=0.01479
[Ridge] BA: CV_MAE=0.02482, Test_MAE=0.01693
[Ridge] BAC: CV_MAE=0.01691, Test_MAE=0.01200
[Ridge] BKNG: CV_MAE=0.01719, Test_MAE=0.01414
[Ridge] BLK: CV_MAE=0.01493, Test_MAE=0.01189
[Ridge] BRK-B: CV_MAE=0.02033, Test_MAE=0.00725
[Ridge] BSX: CV_MAE=0.02058, Test_MAE=0.01192
[Ridge] BX: CV_MAE=0.02307, 

# Random Forest 

In [68]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer

In [69]:
from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE = 42

def bayes_opt_rf_time_series(X_train, y_train, X_test, y_test,
                             n_splits=5,
                             n_iter=30):
    """
    Optimización bayesiana de hiperparámetros para RandomForest
    usando TimeSeriesSplit en el conjunto de entrenamiento.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)

    rf = RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    # Espacio de búsqueda bayesiana
    search_spaces = {
        "n_estimators": Integer(50, 400),
        "max_depth": Integer(3, 20),
        "max_features": Real(0.3, 1.0),
        "min_samples_leaf": Integer(1, 20),
        "min_samples_split": Integer(2, 30)
    }

    opt = BayesSearchCV(
        estimator=rf,
        search_spaces=search_spaces,
        n_iter=n_iter,
        cv=tscv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0
    )

    # Entrenamos SOLO con el train (respetando temporalidad)
    opt.fit(X_train, y_train)

    best_model = opt.best_estimator_
    best_params = opt.best_params_
    cv_mae_mean = -opt.best_score_
    cv_mae_std = opt.cv_results_["std_test_score"][opt.best_index_]

    # Evaluamos en el test (último tramo temporal)
    y_pred = best_model.predict(X_test)
    test_mae = mean_absolute_error(y_test, y_pred)
    test_rmse = rmse(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    resultados = {
        "modelo": best_model,
        "best_params": best_params,
        "cv_mae_mean": cv_mae_mean,
        "cv_mae_std": cv_mae_std,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2
    }
    return resultados

In [70]:
def correr_rf_bayes_en_todos_los_tickers(df,
                                         feature_cols,
                                         target_col="ret_5d",
                                         test_size_frac=0.2,
                                         n_splits=5,
                                         n_iter=30):
    resultados = {}
    tickers = df["Ticker"].unique()
    print(f"Voy a entrenar RF (BayesSearch) para {len(tickers)} tickers")

    for ticker in tickers:
        try:
            X_train, X_test, y_train, y_test, df_ticker = preparar_datos_ticker(
                df,
                ticker=ticker,
                feature_cols=feature_cols,
                target_col=target_col,
                test_size_frac=test_size_frac
            )

            res = bayes_opt_rf_time_series(
                X_train, y_train, X_test, y_test,
                n_splits=n_splits,
                n_iter=n_iter
            )

            resultados[ticker] = res

            print(
                f"[RF Bayes] {ticker}: "
                f"CV_MAE={res['cv_mae_mean']:.5f}, "
                f"Test_MAE={res['test_mae']:.5f}, "
                f"Test_R2={res['test_r2']:.4f}, "
                f"Best={res['best_params']}"
            )


        except ValueError as e:
            # Por si algún ticker tiene pocos datos
            print(f"Saltando {ticker}: {e}")

    return resultados

In [71]:
resultados_rf_bayes = correr_rf_bayes_en_todos_los_tickers(
    df_ml,
    feature_cols=feature_cols,
    target_col=TARGET_COL,
    test_size_frac=0.2,
    n_splits=5,
    n_iter=30  # puedes bajar a 15–20 si se tarda mucho
)

Voy a entrenar RF (BayesSearch) para 116 tickers
[RF Bayes] AAPL: CV_MAE=0.01876, Test_MAE=0.01576, Test_R2=0.6903, Best=OrderedDict({'max_depth': 20, 'max_features': 0.8304553818937872, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50})
[RF Bayes] ABBV: CV_MAE=0.01895, Test_MAE=0.01459, Test_R2=0.5282, Best=OrderedDict({'max_depth': 19, 'max_features': 0.8043547858463951, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 400})


c:\Users\ENRIQUE\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\ma\core.py:2892: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


[RF Bayes] ABT: CV_MAE=0.01252, Test_MAE=0.01083, Test_R2=0.6618, Best=OrderedDict({'max_depth': 20, 'max_features': 0.7825595593048937, 'min_samples_leaf': 3, 'min_samples_split': 2, 'n_estimators': 50})
[RF Bayes] ACN: CV_MAE=0.01659, Test_MAE=0.01576, Test_R2=0.7292, Best=OrderedDict({'max_depth': 20, 'max_features': 0.5383482233854932, 'min_samples_leaf': 6, 'min_samples_split': 2, 'n_estimators': 50})
[RF Bayes] ADBE: CV_MAE=0.02244, Test_MAE=0.01415, Test_R2=0.7460, Best=OrderedDict({'max_depth': 12, 'max_features': 0.5151775856268738, 'min_samples_leaf': 9, 'min_samples_split': 2, 'n_estimators': 54})
[RF Bayes] ADI: CV_MAE=0.02272, Test_MAE=0.01601, Test_R2=0.6069, Best=OrderedDict({'max_depth': 11, 'max_features': 0.9052912821108825, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 210})
[RF Bayes] AMAT: CV_MAE=0.02646, Test_MAE=0.02377, Test_R2=0.7289, Best=OrderedDict({'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_esti

c:\Users\ENRIQUE\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\ma\core.py:2892: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


[RF Bayes] BKNG: CV_MAE=0.01648, Test_MAE=0.01365, Test_R2=0.5821, Best=OrderedDict({'max_depth': 17, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 30, 'n_estimators': 369})
[RF Bayes] BLK: CV_MAE=0.01486, Test_MAE=0.01235, Test_R2=0.6644, Best=OrderedDict({'max_depth': 15, 'max_features': 0.9575588163219402, 'min_samples_leaf': 4, 'min_samples_split': 7, 'n_estimators': 330})
[RF Bayes] BRK-B: CV_MAE=0.01092, Test_MAE=0.00734, Test_R2=0.7584, Best=OrderedDict({'max_depth': 4, 'max_features': 0.9664979190240812, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 400})


...

[RF Bayes] NG=F: CV_MAE=0.04118, Test_MAE=0.04152, Test_R2=0.6835, Best=OrderedDict({'max_depth': 9, 'max_features': 0.9752602594774404, 'min_samples_leaf': 10, 'min_samples_split': 3, 'n_estimators': 204})
[RF Bayes] NOW: CV_MAE=0.02399, Test_MAE=0.01602, Test_R2=0.6615, Best=OrderedDict({'max_depth': 8, 'max_features': 1.0, 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 349})


c:\Users\ENRIQUE\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\ma\core.py:2892: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


[RF Bayes] NVDA: CV_MAE=0.03457, Test_MAE=0.02124, Test_R2=0.5801, Best=OrderedDict({'max_depth': 19, 'max_features': 1.0, 'min_samples_leaf': 8, 'min_samples_split': 2, 'n_estimators': 109})
[RF Bayes] ORCL: CV_MAE=0.02327, Test_MAE=0.03887, Test_R2=0.4943, Best=OrderedDict({'max_depth': 3, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50})
[RF Bayes] PANW: CV_MAE=0.02204, Test_MAE=0.01638, Test_R2=0.7945, Best=OrderedDict({'max_depth': 15, 'max_features': 0.4845026197760264, 'min_samples_leaf': 9, 'min_samples_split': 3, 'n_estimators': 331})
[RF Bayes] PEP: CV_MAE=0.01216, Test_MAE=0.01469, Test_R2=0.6308, Best=OrderedDict({'max_depth': 3, 'max_features': 0.8035576111157676, 'min_samples_leaf': 10, 'min_samples_split': 11, 'n_estimators': 400})
[RF Bayes] PFE: CV_MAE=0.01480, Test_MAE=0.01917, Test_R2=0.6236, Best=OrderedDict({'max_depth': 20, 'max_features': 0.7520815731173434, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 378}

c:\Users\ENRIQUE\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\ma\core.py:2892: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


[RF Bayes] SI=F: CV_MAE=0.01973, Test_MAE=0.01900, Test_R2=0.6150, Best=OrderedDict({'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 11, 'min_samples_split': 2, 'n_estimators': 400})
[RF Bayes] SPGI: CV_MAE=0.01212, Test_MAE=0.01227, Test_R2=0.6145, Best=OrderedDict({'max_depth': 9, 'max_features': 0.7136785986639509, 'min_samples_leaf': 3, 'min_samples_split': 2, 'n_estimators': 50})
[RF Bayes] SPY: CV_MAE=0.00983, Test_MAE=0.00513, Test_R2=0.7634, Best=OrderedDict({'max_depth': 15, 'max_features': 0.9575588163219402, 'min_samples_leaf': 4, 'min_samples_split': 7, 'n_estimators': 330})
[RF Bayes] SYK: CV_MAE=0.01170, Test_MAE=0.01178, Test_R2=0.6609, Best=OrderedDict({'max_depth': 17, 'max_features': 0.7529980716134981, 'min_samples_leaf': 6, 'min_samples_split': 2, 'n_estimators': 276})
[RF Bayes] T: CV_MAE=0.01271, Test_MAE=0.01096, Test_R2=0.7008, Best=OrderedDict({'max_depth': 7, 'max_features': 0.5553939876579572, 'min_samples_leaf': 3, 'min_samples_split': 29, 'n_estim

# Prediccion